<a href="https://colab.research.google.com/github/mahatomic/Simulative/blob/Python/%D0%9F%D0%B5%D1%80%D0%B2%D1%8B%D0%B9_%D0%BA%D0%B5%D0%B9%D1%81.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1 кейс

**Ваша задача написать функцию `process_files`, которая принимает на вход два пути к папкам. Из первой папки необходимо выбрать все "чеки" (файлы по шаблону из условия), а во вторую папку сохранить один объединенный чек (отсортированный по дате, а затем по продукту) под названием `combined_data.csv`.**

**Важно**

Перед началом решения выполните следующую ячейку, чтобы загрузить папку с файлами. После выполнения, в папке `reports_main` будут храниться все присланные магазинами чеки.

In [59]:
!wget https://github.com/vs8th/reports/archive/main.zip

import zipfile

with zipfile.ZipFile("main.zip", 'r') as zip_ref:
    zip_ref.extractall("/content")

!rm main.zip

--2026-07-14 19:11:50--  https://github.com/vs8th/reports/archive/main.zip
Resolving github.com (github.com)... 20.27.177.113
Connecting to github.com (github.com)|20.27.177.113|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://codeload.github.com/Vs8th/reports/zip/refs/heads/main [following]
--2026-07-14 19:11:51--  https://codeload.github.com/Vs8th/reports/zip/refs/heads/main
Resolving codeload.github.com (codeload.github.com)... 20.27.177.114
Connecting to codeload.github.com (codeload.github.com)|20.27.177.114|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/zip]
Saving to: ‘main.zip’

main.zip                [ <=>                ]   3.22K  --.-KB/s    in 0s      

2026-07-14 19:11:51 (41.3 MB/s) - ‘main.zip’ saved [3296]



Чтобы посмотреть как выглядят подходящие для объединения чеки выполните следующую ячейку.

In [ ]:
import pandas as pd

df = pd.read_csv('reports-main/2023-02-17-05-38-2.csv', sep=";")
df

,date,product,store,cost
0,2023-02-17,product_0,store_2,10
1,2023-02-17,product_1,store_2,20
2,2023-02-17,product_2,store_2,30
3,2023-02-17,product_3,store_2,40
4,2023-02-17,product_4,store_2,50


**Решение**

Напишите свое решение ниже

In [60]:
from glob import glob
import re
import os

In [3]:
files = glob('reports-main/*.csv')
files

['reports-main/Year 2011-2012-Tаблица 1.csv',
 'reports-main/shop_1.csv',
 'reports-main/2023-02-16-02-55-1.csv',
 'reports-main/Year 2010-2011-Tаблица 1.csv',
 'reports-main/check.csv',
 'reports-main/check_from_3#shop.csv',
 'reports-main/2023-02-19-11-05-4.csv',
 'reports-main/Year 2012-2013-Tаблица 7.csv',
 'reports-main/2023-02-17-05-38-2.csv',
 'reports-main/2023-02-15-10-26-0.csv',
 'reports-main/2023-02-30-02-55-23.csv',
 'reports-main/2023-02-18-19-47-3.csv',
 'reports-main/total_income_for_day.csv']

In [52]:
new_files = []

for file in files:
  refiles = re.match(r'.*\d{4}-\d{2}-\d{2}-\d{2}-\d{2}-\d+.csv',file)
  if refiles:
    new_files.append(refiles.group())

new_files

['reports-main/2023-02-16-02-55-1.csv',
 'reports-main/2023-02-19-11-05-4.csv',
 'reports-main/2023-02-17-05-38-2.csv',
 'reports-main/2023-02-15-10-26-0.csv',
 'reports-main/2023-02-30-02-55-23.csv',
 'reports-main/2023-02-18-19-47-3.csv']

**Примечание**

Не все файлы подходящие по названию, будут подходить по содержанию. Там может оказаться лишний столбец, например. Ориентируйтесь на столбцы из чека выше - это то, что вас интересует. Остальные столбцы можно просто отбросить.

**Важно**: разделителем файла на выходе должна быть запятая.

In [61]:
def process_files(src_folder, dest_folder):
  files = glob(src_folder + '/*.csv')

  new_files = []

  for file in files:
    refiles = re.match(r'.*\d{4}-\d{2}-\d{2}-\d{2}-\d{2}-\d+.csv',file)
    if refiles:
      new_files.append(refiles.group())

  if not os.path.exists(dest_folder):
      os.makedirs(dest_folder)

  header_line = None
  all_lines = []

  for k in range(len(new_files)):
    with open(new_files[k], 'r') as src:
      data = src.readlines()
      data0 = data[0].strip().split(';')
      ind = []
      for i, word in enumerate(data0):
        if word.strip() in ['date', 'product', 'store', 'cost']:
          ind.append(i)

      if k == 0:
        header_line = ','.join([data0[i] for i in ind])

      data = data[1:]

      for str in data:
        split_str = str.strip().split(';')
        str_list = []
        for i in ind:
          str_list.append(split_str[i])
        all_lines.append(str_list)

  all_lines.sort(key=lambda row: (row[0], row[1]))

  with open(os.path.join(dest_folder, 'combined_data.csv'), 'w') as dest:
    dest.write(header_line + '\n')
    for line in all_lines:
        dest.write(','.join(line) + '\n')

src_folder = 'reports-main'
dest_folder = 'comb_reports'
process_files(src_folder, dest_folder)

✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [62]:
# Здесь будет скачиваться файл с эталонным ответом

!wget https://gist.github.com/Vs8th/9347dd7b8f59de2997feb19770dc32c1/raw/data.csv

import pandas as pd

user_answer = pd.read_csv(f'{dest_folder}/combined_data.csv')
correct_answer = pd.read_csv('data.csv')

--2026-07-14 19:12:11--  https://gist.github.com/Vs8th/9347dd7b8f59de2997feb19770dc32c1/raw/data.csv
Resolving gist.github.com (gist.github.com)... 20.27.177.113
Connecting to gist.github.com (gist.github.com)|20.27.177.113|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/9347dd7b8f59de2997feb19770dc32c1/raw/data.csv [following]
--2026-07-14 19:12:12--  https://gist.githubusercontent.com/Vs8th/9347dd7b8f59de2997feb19770dc32c1/raw/data.csv
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 984 [text/plain]
Saving to: ‘data.csv’

data.csv            100%[===================>]     984  --.-KB/s    in 0s      

2026-07-14 19:12:12 (43.7 MB/s) - ‘data.csv’ saved [984/984]



In [63]:
user_answer

,date,product,store,cost
0,2023-02-15,product_0,store_0,10
1,2023-02-15,product_1,store_0,20
2,2023-02-15,product_2,store_0,30
3,2023-02-15,product_3,store_0,40
4,2023-02-15,product_4,store_0,50
5,2023-02-16,product_0,store_1,10
6,2023-02-16,product_1,store_1,20
7,2023-02-16,product_2,store_1,30
8,2023-02-16,product_3,store_1,40
9,2023-02-16,product_4,store_1,50


In [32]:
correct_answer

,date,product,store,cost
0,2023-02-15,product_0,store_0,10
1,2023-02-15,product_1,store_0,20
2,2023-02-15,product_2,store_0,30
3,2023-02-15,product_3,store_0,40
4,2023-02-15,product_4,store_0,50
5,2023-02-16,product_0,store_1,10
6,2023-02-16,product_1,store_1,20
7,2023-02-16,product_2,store_1,30
8,2023-02-16,product_3,store_1,40
9,2023-02-16,product_4,store_1,50


In [64]:
try:
  assert (user_answer == correct_answer).all().all(), 'Ответы не совпадают'
  assert user_answer.columns.equals(correct_answer.columns), 'Названия столбцов не совпадают'
except Exception as err:
  raise AssertionError(f'При проверке возникла ошибка {repr(err)}')
else:
  print('Поздравляем, Вы справились и успешно прошли все проверки!')

Поздравляем, Вы справились и успешно прошли все проверки!
